# Treinamento: ResNet + BiLSTM

Prediz um *arousal score* contínuo por clipe a partir do vídeo em grayscale.
Cada experimento gera um diretório próprio em `runs/` para o TensorBoard.

**Decisões de treino:**
- `epochs=100` (teto pq tem *early stopping*).
- Checkpoint/early-stopping por **CCC**; scheduler observa a **val loss**.
- `batch_size=4` (`2` na ResNet50 por memória).
- *Gradient clipping* `1.0` e semente fixa.

## 1. Setup

In [1]:
import os
import sys
import datetime
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

from resnetlstm import ResNetLSTM
from loaders import build_video_dataloader
from balanced_dataset import balanced_df

In [2]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 435
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch: 2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA disponível: True
CUDA: 12.1
GPU: NVIDIA RTX A4500
Device: cuda


## 2. Configuração

Metadados a serem ajustados.

In [3]:
# paths
LABELS_ALL   = "/mnt/storage_C4/gaussian_football/data/labels/labels_all.csv"
LABELS_DIR   = "/mnt/storage_C4/gaussian_football/data/labels"
CKPT_DIR     = "/mnt/storage_C4/gaussian_football/models/checkpoints"
RUNS_DIR     = "/mnt/storage_C4/gaussian_football/runs"  # logs do TensorBoard

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# treino 
EPOCHS        = 100        # teto por causa do early stopping decide
PATIENCE      = 10         # épocas sem melhora antes de parar
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
MONITOR       = "ccc"      # métrica de checkpoint/early-stopping: "ccc" | "mae" | "loss"

# dataloaders 
BATCH_SIZE = 2

# modelo padrão
MODEL_DEFAULTS = dict(
    frame_step=2,
    hidden_size=256,
    num_layers=1,
    use_dropout=False,
    dropout_p=0.3,
)

## 3. Balanceando os Dados

Gera os CSVs balanceados (50% highlight / 50% não-highlight por partida,
com `threshold=0.5` no `arousal_score`). Só recalcula se os arquivos não existirem
(use `FORCE_REBUILD = True` para refazer).

In [4]:
FORCE_REBUILD = False

paths_balanced = {
    "train": os.path.join(LABELS_DIR, "balanced_labels_train.csv"),
    "valid": os.path.join(LABELS_DIR, "balanced_labels_val.csv"),
    "test":  os.path.join(LABELS_DIR, "balanced_labels_test.csv"),
}

if FORCE_REBUILD or not all(os.path.exists(p) for p in paths_balanced.values()):
    all_data = pd.read_csv(LABELS_ALL)
    for split, out_path in paths_balanced.items():
        subset = all_data[all_data["split"] == split]
        balanced = balanced_df(subset, "game_id", threshold=0.5, random_state=SEED)
        balanced.to_csv(out_path, index=False)
        print(f"{split}: {len(balanced)} clipes -> {out_path}")
else:
    print("CSVs balanceados já existem. (FORCE_REBUILD=True para refazer.)")

CSVs balanceados já existem. (FORCE_REBUILD=True para refazer.)


## 4. DataLoaders

In [5]:
TRAIN_PATH = f"{LABELS_DIR}/balanced_labels_train.csv"
VAL_PATH   = f"{LABELS_DIR}/balanced_labels_val.csv"

train_video_loader = build_video_dataloader(
    csv_path=TRAIN_PATH,
    split="train",
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

valid_video_loader = build_video_dataloader(
    csv_path=VAL_PATH,
    split="valid",
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

Dataset: 3034/3034 exemplos válidos.
Dataset: 1322/1322 exemplos válidos.


## 5. Métricas

O **CCC** (Concordance Correlation Coefficient) mede concordância entre predição e alvo,
punindo tanto erro de correlação quanto deslocamento de escala/média, por isso é sensível
ao *variance collapse* (modelo prevendo sempre perto da média).

$$\rho_c = \dfrac{2\,\rho\,\sigma_x\,\sigma_y}{\sigma_x^2 + \sigma_y^2 + (\mu_x - \mu_y)^2}$$

In [6]:
def calculate_ccc(y_true, y_pred):
    '''Concordance Correlation Coefficient sobre dois arrays 1D.'''
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true,  var_pred  = np.var(y_true),  np.var(y_pred)
    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2
    return 0.0 if denominator == 0 else (2 * covariance) / denominator

## 6. Funções de perda

- `mse`: minimiza o erro quadrático médio.
- `bce`: `BCEWithLogitsLoss`, trata o arousal score como probabilidade em `[0, 1]`, já tem sigmoide embutida.
- `ccc`: `1 - CCC`, força o modelo a preservar variância e correlação.
- `combined`: `MSE + alpha * (1 - CCC)`, combinando as coisas.

In [7]:
def ccc_loss(pred, target):
    pred_mean, target_mean = pred.mean(), target.mean()
    covariance = ((pred - pred_mean) * (target - target_mean)).mean()
    pred_var, target_var = pred.var(), target.var()
    ccc = (2 * covariance) / (pred_var + target_var + (pred_mean - target_mean) ** 2 + 1e-8)
    return 1 - ccc

def combined_loss(pred, target, alpha=0.5):
    return nn.MSELoss()(pred, target) + alpha * ccc_loss(pred, target)

LOSSES = {
    "mse":      nn.MSELoss(),
    "bce":      nn.BCEWithLogitsLoss(),
    "ccc":      ccc_loss,
    "combined": combined_loss,
}

## 7. Treino

Função de treino unificada.

In [8]:
MONITOR_MODE = {"ccc": "max", "mae": "min", "loss": "min"}

def _apply_freeze(model, freeze_backbone):
    '''Liga/desliga o gradiente da ResNet. conv1 (índice 0) fica sempre treinável.'''
    for name, param in model.cnn.named_parameters():
        param.requires_grad = name.startswith("0.") or (not freeze_backbone)

def _set_backbone_eval(model):
    '''Congela estatísticas de BatchNorm na ResNet. Conv1 fica em train.'''
    for i, module in enumerate(model.cnn):
        module.train() if i == 0 else module.eval()

def _is_better(current, best, mode):
    return current > best if mode == "max" else current < best

def _log_pred_scatter(writer, y_true, y_pred, tag, step):
    '''Scatter pred x target — faixa horizontal achatada = variance collapse.'''
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(y_true, y_pred, s=8, alpha=0.4)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)  # diagonal ideal
    ax.set_xlabel("target"); ax.set_ylabel("pred"); ax.set_title("pred vs target")
    writer.add_figure(tag, fig, step)
    plt.close(fig)


def train_model(
    model, train_loader, val_loader, *,
    criterion, run_name, optimizer, scheduler,
    backbone_name, loss_name,
    freeze_mode="finetune", unfreeze_epoch=5,
    epochs=EPOCHS, patience=PATIENCE, monitor=MONITOR,
    grad_clip=GRAD_CLIP, use_amp=False,
    checkpoint_path=None, device=device,
):
    '''Treina e valida por época, logando tudo no TensorBoard.

    Args:
        criterion: função de perda (vindo de LOSSES).
        run_name: nome do experimento (usado no log_dir e no checkpoint).
        backbone_name, loss_name: usados nos hparams do TensorBoard.
        freeze_mode: "full" | "frozen" | "finetune".
        unfreeze_epoch: época de descongelamento (só vale para "finetune").
        monitor: métrica de checkpoint/early-stopping ("ccc" | "mae" | "loss").
        use_amp: ativa mixed precision (mais rápido; cheque o ccc_loss em fp16).
    Returns:
        dict com as melhores métricas e o caminho do checkpoint.
    '''
    model.to(device)
    torch.cuda.empty_cache()
    if checkpoint_path is None:
        checkpoint_path = os.path.join(CKPT_DIR, f"{run_name}.pth")
    last_path = os.path.join(CKPT_DIR, f"{run_name}_last.pth")

    # com BCE a saída é logit: aplica sigmoid antes das métricas (não na loss)
    use_sigmoid = isinstance(criterion, nn.BCEWithLogitsLoss)

    mode = MONITOR_MODE[monitor]
    best_score = -np.inf if mode == "max" else np.inf
    best_metrics, best_true, best_pred = {}, None, None
    epochs_no_improve = 0

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(RUNS_DIR, f"{run_name}_{stamp}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):
        # estado de congelamento desta época
        if freeze_mode == "full":
            backbone_frozen = False
        elif freeze_mode == "frozen":
            backbone_frozen = True
        else:  # finetune
            backbone_frozen = epoch < unfreeze_epoch
            if epoch == unfreeze_epoch:
                print(f"[época {epoch+1}] descongelando a ResNet (fine-tuning completo)")

        _apply_freeze(model, backbone_frozen)

        # ----- treino -----
        model.train()
        if backbone_frozen:
            _set_backbone_eval(model)

        train_loss = 0.0
        train_true, train_pred = [], []
        for videos, masks, targets in tqdm(train_loader, desc=f"época {epoch+1}/{epochs} [treino]", leave=False):
            videos = videos.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True).float().view(-1)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(videos).reshape(-1)
                loss = criterion(outputs, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)  # unscale antes do clip
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * videos.size(0)
            preds = torch.sigmoid(outputs) if use_sigmoid else outputs
            train_true.extend(targets.detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())
        train_loss /= len(train_loader.dataset)

        train_true, train_pred = np.array(train_true), np.array(train_pred)
        train_mae = np.mean(np.abs(train_true - train_pred))
        train_ccc = calculate_ccc(train_true, train_pred)

        # ----- validação -----
        model.eval()
        val_loss = 0.0
        all_true, all_pred = [], []
        with torch.no_grad():
            for videos, masks, targets in tqdm(val_loader, desc=f"época {epoch+1}/{epochs} [val]", leave=False):
                videos = videos.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True).float().view(-1)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    outputs = model(videos).reshape(-1)
                    val_loss += criterion(outputs, targets).item() * videos.size(0)
                preds = torch.sigmoid(outputs) if use_sigmoid else outputs
                all_true.extend(targets.cpu().numpy())
                all_pred.extend(preds.cpu().numpy())
        val_loss /= len(val_loader.dataset)

        all_true, all_pred = np.array(all_true), np.array(all_pred)
        val_mae = np.mean(np.abs(all_true - all_pred))
        val_ccc = calculate_ccc(all_true, all_pred)
        pred_std = float(np.std(all_pred)) 
        target_std = float(np.std(all_true))

        scheduler.step(val_loss)

        # ----- tensorboard -----
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Metrics/MAE_train", train_mae, epoch)
        writer.add_scalar("Metrics/MAE_val", val_mae, epoch)
        writer.add_scalar("Metrics/CCC_train", train_ccc, epoch)
        writer.add_scalar("Metrics/CCC_val", val_ccc, epoch)
        writer.add_scalar("Diagnostics/pred_std", pred_std, epoch)
        writer.add_scalar("Diagnostics/target_std", target_std, epoch)
        writer.add_histogram("Val/predictions", all_pred, epoch)
        for gi, pg in enumerate(optimizer.param_groups):
            writer.add_scalar(f"Train/LR_group{gi}", pg["lr"], epoch)

        print(
            f"época [{epoch+1}/{epochs}]"
            f" | loss  train {train_loss:.4f}  val {val_loss:.4f}"
            f" | MAE   train {train_mae:.4f}  val {val_mae:.4f}"
            f" | CCC   train {train_ccc:.4f}  val {val_ccc:.4f}"
            f" | pred_std {pred_std:.4f} | LR {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ----- checkpoint (best + last) + early stopping -----
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "run_name": run_name}, last_path)

        current = {"ccc": val_ccc, "mae": val_mae, "loss": val_loss}[monitor]
        if _is_better(current, best_score, mode):
            best_score = current
            best_metrics = {"val_loss": val_loss, "val_mae": val_mae,
                            "val_ccc": val_ccc, "epoch": epoch}
            best_true, best_pred = all_true, all_pred
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_metrics": best_metrics,
                "run_name": run_name,
            }, checkpoint_path)
            print(f"  novo melhor ({monitor}={best_score:.4f}) salvo em {checkpoint_path}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"early stopping (sem melhora por {patience} épocas)")
                break

    # scatter do melhor epoch — diagnóstico de variance collapse
    if best_pred is not None:
        _log_pred_scatter(writer, best_true, best_pred, "Val/pred_vs_target", best_metrics["epoch"])

    # hparams para comparar runs na aba HPARAMS (run_name="." liga aos scalars)
    writer.add_hparams(
        {"backbone": backbone_name, "loss": loss_name, "freeze_mode": freeze_mode,
         "lr": LR, "hidden_size": MODEL_DEFAULTS["hidden_size"],
         "num_layers": MODEL_DEFAULTS["num_layers"], "frame_step": MODEL_DEFAULTS["frame_step"]},
        {"best/val_ccc": best_metrics.get("val_ccc", 0.0),
         "best/val_mae": best_metrics.get("val_mae", 0.0),
         "best/val_loss": best_metrics.get("val_loss", 0.0)},
        run_name=".",
    )
    writer.close()
    print(f"concluído. melhor {monitor}: {best_score:.4f}")
    return {"checkpoint": checkpoint_path, "best_metrics": best_metrics}

## 8. run_experiment

Função reaproveitável.

In [9]:
def run_experiment(
    backbone_name="resnet18",
    loss_name="bce",
    freeze_mode="finetune",
    unfreeze_epoch=5,
    lr=LR,
    lr_backbone=None,
    weight_decay=WEIGHT_DECAY,
    model_kwargs=None,
    epochs=EPOCHS,
    patience=PATIENCE,
    monitor=MONITOR,
    use_amp=False,
):
    '''Roda um experimento completo e retorna o resultado.'''
    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}
    run_name = f"{backbone_name}__{loss_name}__{freeze_mode}"
    print(f"\n=== {run_name} ===")

    model = ResNetLSTM(backbone_name=backbone_name, **model_kwargs).to(device)

    if lr_backbone is None:
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = AdamW([
            {"params": model.cnn.parameters(),  "lr": lr_backbone},
            {"params": model.lstm.parameters(), "lr": lr},
            {"params": model.head.parameters(), "lr": lr},
        ], weight_decay=weight_decay)

    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

    return train_model(
        model, train_video_loader, valid_video_loader,
        criterion=LOSSES[loss_name], run_name=run_name,
        optimizer=optimizer, scheduler=scheduler,
        backbone_name=backbone_name, loss_name=loss_name,
        freeze_mode=freeze_mode, unfreeze_epoch=unfreeze_epoch,
        epochs=epochs, patience=patience, monitor=monitor,
        use_amp=use_amp,
    )

## 9. Rodar experimentos

Altos experimentos.

In [ ]:
results = {}

experimentos = (
    [("resnet18", loss, "frozen", 5) for loss in ["mse", "bce", "ccc"]]
    + [(backbone, loss, "frozen", 5) for backbone in ["resnet34", "resnet50"] for loss in ["mse", "bce", "ccc"]]
    + [("resnet18", loss, "finetune", 5) for loss in ["mse", "bce", "ccc"]]
)

for backbone, loss_name, freeze_mode, unfreeze_epoch in experimentos:
    key = f"{backbone}__{loss_name}__{freeze_mode}"
    print(f"\n>>> iniciando {key}")
    try:
        results[key] = run_experiment(
            backbone_name=backbone,
            loss_name=loss_name,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=100,
        )
    except Exception as e:
        print(f"ERRO em {key}: {e}")
        results[key] = None

print("\n=== RESUMO ===")
for key, result in results.items():
    if result is None:
        print(f"{key}: FALHOU")
    else:
        m = result["best_metrics"]
        print(f"{key}: CCC={m['val_ccc']:.4f} | MAE={m['val_mae']:.4f} | loss={m['val_loss']:.4f}")


>>> iniciando resnet18__mse__frozen

=== resnet18__mse__frozen ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs/resnet18__mse__frozen_20260617-192946


época 1/100 [treino]:   0%|          | 0/1517 [00:00<?, ?it/s]

ERRO em resnet18__mse__frozen: Caught ImportError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/mnt/storage_C4/gaussian_football/preprocessing/datasets_mel_video.py", line 152, in __getitem__
    video, _, _ = torchvision.io.read_video(row["clip_path"], pts_unit="sec")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torchvision/io/video.py", line 291, in read_video
    _check_av_available()
  File "/home/al.pedro.santos

época 1/100 [treino]:   0%|          | 0/1517 [00:00<?, ?it/s]

ERRO em resnet18__bce__frozen: Caught ImportError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/mnt/storage_C4/gaussian_football/preprocessing/datasets_mel_video.py", line 152, in __getitem__
    video, _, _ = torchvision.io.read_video(row["clip_path"], pts_unit="sec")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torchvision/io/video.py", line 291, in read_video
    _check_av_available()
  File "/home/al.pedro.santos

...

ERRO em resnet18__ccc__frozen: Caught ImportError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/mnt/storage_C4/gaussian_football/preprocessing/datasets_mel_video.py", line 152, in __getitem__
    video, _, _ = torchvision.io.read_video(row["clip_path"], pts_unit="sec")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torchvision/io/video.py", line 291, in read_video
    _check_av_available()
  File "/home/al.pedro.santos

época 1/100 [treino]:   0%|          | 0/1517 [00:00<?, ?it/s]

ERRO em resnet34__mse__frozen: Caught ImportError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/mnt/storage_C4/gaussian_football/preprocessing/datasets_mel_video.py", line 152, in __getitem__
    video, _, _ = torchvision.io.read_video(row["clip_path"], pts_unit="sec")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torchvision/io/video.py", line 291, in read_video
    _check_av_available()
  File "/home/al.pedro.santos

época 1/100 [treino]:   0%|          | 0/1517 [00:00<?, ?it/s]

ERRO em resnet34__bce__frozen: Caught ImportError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/mnt/storage_C4/gaussian_football/preprocessing/datasets_mel_video.py", line 152, in __getitem__
    video, _, _ = torchvision.io.read_video(row["clip_path"], pts_unit="sec")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torchvision/io/video.py", line 291, in read_video
    _check_av_available()
  File "/home/al.pedro.santos

In [10]:
results = {}

experimentos = (
    [(backbone, loss, "frozen", 5) for backbone in ["resnet50"] for loss in ["mse", "bce", "ccc"]]
    + [("resnet18", loss, "finetune", 5) for loss in ["mse", "bce", "ccc"]]
)

for backbone, loss_name, freeze_mode, unfreeze_epoch in experimentos:
    key = f"{backbone}__{loss_name}__{freeze_mode}"
    print(f"\n>>> iniciando {key}")
    try:
        results[key] = run_experiment(
            backbone_name=backbone,
            loss_name=loss_name,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=100,
        )
    except Exception as e:
        print(f"ERRO em {key}: {e}")
        results[key] = None

print("\n=== RESUMO ===")
for key, result in results.items():
    if result is None:
        print(f"{key}: FALHOU")
    else:
        m = result["best_metrics"]
        print(f"{key}: CCC={m['val_ccc']:.4f} | MAE={m['val_mae']:.4f} | loss={m['val_loss']:.4f}")


>>> iniciando resnet50__mse__frozen

=== resnet50__mse__frozen ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs/resnet50__mse__frozen_20260615-204626


época [1/100] | train 0.1978 | val 0.1978 | MAE 0.4277 | CCC -0.0116 | pred_std 0.0433 | LR 1.00e-04
  novo melhor (ccc=-0.0116) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet50__mse__frozen.pth


época [2/100] | train 0.1933 | val 0.1999 | MAE 0.4275 | CCC -0.0098 | pred_std 0.0458 | LR 1.00e-04
  novo melhor (ccc=-0.0098) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet50__mse__frozen.pth


época [3/100] | train 0.1891 | val 0.2004 | MAE 0.4292 | CCC -0.0157 | pred_std 0.0609 | LR 1.00e-04


época [4/100] | train 0.1877 | val 0.2075 | MAE 0.4289 | CCC -0.0116 | pred_std 0.1042 | LR 1.00e-04


...

época [20/100] | train 0.9833 | val 0.9962 | MAE 0.4969 | CCC -0.0000 | pred_std 0.2868 | LR 2.50e-05


época [21/100] | train 0.9582 | val 1.0044 | MAE 0.4981 | CCC -0.0259 | pred_std 0.4060 | LR 2.50e-05


época [22/100] | train 0.9816 | val 0.9996 | MAE 0.4562 | CCC -0.0037 | pred_std 0.2975 | LR 2.50e-05


época [23/100] | train 0.9693 | val 1.0059 | MAE 0.4864 | CCC 0.0080 | pred_std 0.3612 | LR 1.25e-05


época [24/100] | train 0.9755 | val 0.9997 | MAE 0.4994 | CCC -0.0397 | pred_std 0.3931 | LR 1.25e-05
early stopping (sem melhora por 10 épocas)
concluído. melhor ccc: 0.0293

=== RESUMO ===
resnet50__mse__frozen: CCC=0.0061 | MAE=0.4248 | loss=0.2173
resnet50__bce__frozen: CCC=0.0388 | MAE=0.4175 | loss=0.6935
resnet50__ccc__frozen: CCC=0.0833 | MAE=0.4544 | loss=1.0050
resnet18__mse__finetune: CCC=0.0390 | MAE=0.4162 | loss=0.1896
resnet18__bce__finetune: CCC=0.0184 | MAE=0.4212 | loss=0.6780
resnet18__ccc__finetune: CCC=0.0293 | MAE=0.4335 | loss=0.9988


In [13]:
results = {}

experimentos = (
    [(backbone, loss, "finetune", 5) for backbone in ["resnet34","resnet50"]  for loss in ["mse", "bce", "ccc"]]
)

for backbone, loss_name, freeze_mode, unfreeze_epoch in experimentos:
    key = f"{backbone}__{loss_name}__{freeze_mode}"
    print(f"\n>>> iniciando {key}")
    try:
        results[key] = run_experiment(
            backbone_name=backbone,
            loss_name=loss_name,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=100,
        )
    except Exception as e:
        print(f"ERRO em {key}: {e}")
        results[key] = None

print("\n=== RESUMO ===")
for key, result in results.items():
    if result is None:
        print(f"{key}: FALHOU")
    else:
        m = result["best_metrics"]
        print(f"{key}: CCC={m['val_ccc']:.4f} | MAE={m['val_mae']:.4f} | loss={m['val_loss']:.4f}")


>>> iniciando resnet34__mse__finetune

=== resnet34__mse__finetune ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs/resnet34__mse__finetune_20260616-094507


época [1/100] | train 0.2041 | val 0.1983 | MAE 0.4243 | CCC 0.0074 | pred_std 0.0293 | LR 1.00e-04
  novo melhor (ccc=0.0074) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet34__mse__finetune.pth


época [2/100] | train 0.1947 | val 0.1979 | MAE 0.4235 | CCC 0.0077 | pred_std 0.0374 | LR 1.00e-04
  novo melhor (ccc=0.0077) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet34__mse__finetune.pth


época [3/100] | train 0.1944 | val 0.1908 | MAE 0.4203 | CCC 0.0229 | pred_std 0.0357 | LR 1.00e-04
  novo melhor (ccc=0.0229) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet34__mse__finetune.pth


época [4/100] | train 0.1928 | val 0.1960 | MAE 0.4201 | CCC 0.0216 | pred_std 0.0590 | LR 1.00e-04


...

época [14/100] | train 0.6840 | val 0.6866 | MAE 0.4269 | CCC -0.0062 | pred_std 0.0236 | LR 5.00e-05


época [15/100] | train 0.6844 | val 0.6842 | MAE 0.4258 | CCC -0.0006 | pred_std 0.0068 | LR 2.50e-05


época [16/100] | train 0.6834 | val 0.6850 | MAE 0.4263 | CCC -0.0034 | pred_std 0.0189 | LR 2.50e-05


época [17/100] | train 0.6825 | val 0.6830 | MAE 0.4251 | CCC 0.0036 | pred_std 0.0261 | LR 2.50e-05
early stopping (sem melhora por 10 épocas)
concluído. melhor ccc: 0.0103

>>> iniciando resnet50__ccc__finetune

=== resnet50__ccc__finetune ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs/resnet50__ccc__finetune_20260616-160917


KeyboardInterrupt: 

In [10]:
results = {}

experimentos = (
    [(backbone, loss, "finetune", 5) for backbone in ["resnet50"]  for loss in ["ccc"]]
)

for backbone, loss_name, freeze_mode, unfreeze_epoch in experimentos:
    key = f"{backbone}__{loss_name}__{freeze_mode}"
    print(f"\n>>> iniciando {key}")
    try:
        results[key] = run_experiment(
            backbone_name=backbone,
            loss_name=loss_name,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=100,
        )
    except Exception as e:
        print(f"ERRO em {key}: {e}")
        results[key] = None

print("\n=== RESUMO ===")
for key, result in results.items():
    if result is None:
        print(f"{key}: FALHOU")
    else:
        m = result["best_metrics"]
        print(f"{key}: CCC={m['val_ccc']:.4f} | MAE={m['val_mae']:.4f} | loss={m['val_loss']:.4f}")


>>> iniciando resnet50__ccc__finetune

=== resnet50__ccc__finetune ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs/resnet50__ccc__finetune_20260617-193833


época 1/100 [treino]:   0%|          | 0/1517 [00:00<?, ?it/s]

época 1/100 [val]:   0%|          | 0/661 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a630d62160>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a630d62160>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época [1/100] | loss  train 0.9441  val 0.9898 | MAE   train 0.4278  val 0.4368 | CCC   train 0.2211  val 0.2941 | pred_std 0.4008 | LR 1.00e-04
  novo melhor (ccc=0.2941) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet50__ccc__finetune.pth


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a630d62160>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a630d62160>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

época [6/100] | loss  train 0.9555  val 0.9996 | MAE   train 0.6000  val 0.4093 | CCC   train 0.0362  val 0.0883 | pred_std 0.3189 | LR 1.00e-04


época 7/100 [treino]:   0%|          | 0/1517 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a630d62160>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1587, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ambiente_aluno/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a630d62160>
Traceback (most recent call last):
  File "/home/al.pedro.santos/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  Fi

KeyboardInterrupt: 

## 10. TensorBoard

Ver os resultados.

In [11]:
%load_ext tensorboard

In [12]:
%tensorboard --logdir /mnt/storage_C4/gaussian_football/runs/